<a href="https://colab.research.google.com/github/FaridRash/brain-ct-hemorrhage-segmentation/blob/main/04_air_filtering_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Github

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Unzipping

In [2]:
import os

output_dir = '/content/brain_ct_project'
os.makedirs(output_dir, exist_ok=True)

zip_file_path = '/content/drive/MyDrive/brain_ct_project/processed_data_full.zip'

# Unzip the file
!unzip -q "{zip_file_path}" -d "{output_dir}"

print(f"File unzipped to: {output_dir}")
!ls -F "{output_dir}"

File unzipped to: /content/brain_ct_project
test/  train/  val/


In [3]:
import shutil

destination_dir = '/content/brain_ct_project/'

#label_train.csv
source_file = '/content/drive/MyDrive/brain_ct_project/label_train.csv'
shutil.copy(source_file, destination_dir)

#label_val.csv
source_file = '/content/drive/MyDrive/brain_ct_project/label_val.csv'
shutil.copy(source_file, destination_dir)

#label_test.csv
source_file = '/content/drive/MyDrive/brain_ct_project/label_test.csv'
shutil.copy(source_file, destination_dir)


print(f"File '{source_file}' copied to '{destination_dir}'")
!ls -F "{destination_dir}"

File '/content/drive/MyDrive/brain_ct_project/label_test.csv' copied to '/content/brain_ct_project/'
label_test.csv	label_train.csv  label_val.csv	test/  train/  val/


#Air filtering

In [4]:
import numpy as np

def is_air_slice(img, air_hu_threshold=-800, air_ratio_threshold=0.9):
    air_pixels = np.sum(img < air_hu_threshold)
    total_pixels = img.size
    air_ratio = air_pixels / total_pixels
    return air_ratio > air_ratio_threshold

In [5]:
import os
import pandas as pd
import numpy as np

base_path = "/content/brain_ct_project"
splits = ["train", "val", "test"]

for split in splits:

    print(f"\nProcessing {split}...")

    image_dir = os.path.join(base_path, split, "images")
    label_path = os.path.join(base_path, f"label_{split}.csv")

    df = pd.read_csv(label_path)

    keep_rows = []

    for _, row in df.iterrows():

        img_path = os.path.join(image_dir, row["filename"])
        img = np.load(img_path)

        label = row["label"]

        if label == 1:
            keep_rows.append(row)  # ALWAYS keep positives
        else:
            if not is_air_slice(img):
                keep_rows.append(row)

    new_df = pd.DataFrame(keep_rows)

    new_df.to_csv(
        os.path.join(base_path, f"label_{split}_filtered.csv"),
        index=False
    )

    print(f"{split}: {len(df)} → {len(new_df)} kept")


Processing train...
train: 2281 → 2116 kept

Processing val...
val: 241 → 224 kept

Processing test...
test: 292 → 274 kept


In [6]:
print("Before filtering:")
print(df["label"].value_counts())

print("After filtering:")
print(new_df["label"].value_counts())

Before filtering:
label
0    255
1     37
Name: count, dtype: int64
After filtering:
label
0    237
1     37
Name: count, dtype: int64


#remove filtered images

In [7]:
import os
import pandas as pd

base_path = "/content/brain_ct_project"
splits = ["train", "val", "test"]

for split in splits:

    print(f"\nCleaning {split}...")

    image_dir = os.path.join(base_path, split, "images")
    filtered_csv_path = os.path.join(base_path, f"label_{split}_filtered.csv")

    df = pd.read_csv(filtered_csv_path)

    # set of files we WANT to keep
    keep_files = set(df["filename"].values)

    removed_count = 0

    for img_name in os.listdir(image_dir):

        if img_name not in keep_files:
            os.remove(os.path.join(image_dir, img_name))
            removed_count += 1

    print(f"{split}: removed {removed_count} files")


Cleaning train...
train: removed 165 files

Cleaning val...
val: removed 17 files

Cleaning test...
test: removed 18 files


In [8]:
for split in splits:

    image_dir = os.path.join(base_path, split, "images")
    filtered_csv_path = os.path.join(base_path, f"label_{split}_filtered.csv")

    df = pd.read_csv(filtered_csv_path)

    print(f"\n{split}")
    print("Images:", len(os.listdir(image_dir)))
    print("CSV:", len(df))


train
Images: 2116
CSV: 2116

val
Images: 224
CSV: 224

test
Images: 274
CSV: 274


#Remove old zip file from Google Drive

In [9]:
import os

zip_path = "/content/drive/MyDrive/brain_ct_project/processed_data_full.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)
    print("✅ Zip file removed successfully")
else:
    print("❌ File not found")

✅ Zip file removed successfully


In [20]:
zip_path = "/content/drive/MyDrive/brain_ct_project/label_test.csv"

if os.path.exists(zip_path):
    os.remove(zip_path)
    print("✅ Zip file removed successfully")
else:
    print("❌ File not found")

✅ Zip file removed successfully


In [21]:
zip_path = "/content/drive/MyDrive/brain_ct_project/label_train.csv"

if os.path.exists(zip_path):
    os.remove(zip_path)
    print("✅ Zip file removed successfully")
else:
    print("❌ File not found")

✅ Zip file removed successfully


In [22]:
zip_path = "/content/drive/MyDrive/brain_ct_project/label_val.csv"

if os.path.exists(zip_path):
    os.remove(zip_path)
    print("✅ Zip file removed successfully")
else:
    print("❌ File not found")

✅ Zip file removed successfully


In [10]:
print(os.path.exists(zip_path))

False


#Organize the files and zip the output

In [11]:
import shutil
import os

base_path = "/content/brain_ct_project"

splits = ["train", "val", "test"]

for split in splits:

    src = os.path.join(base_path, f"label_{split}_filtered.csv")
    dst = os.path.join(base_path, f"label_{split}.csv")

    shutil.move(src, dst)

print("✅ Filtered labels replaced")

✅ Filtered labels replaced


In [12]:
for file in os.listdir(base_path):
    if "filtered" in file:
        os.remove(os.path.join(base_path, file))

In [13]:
import shutil

for split in splits:
    mask_path = os.path.join(base_path, split, "masks")

    if os.path.exists(mask_path):
        shutil.rmtree(mask_path)

print("✅ Masks removed")

✅ Masks removed


In [14]:
for split in splits:
    img_count = len(os.listdir(os.path.join(base_path, split, "images")))
    csv_count = len(pd.read_csv(os.path.join(base_path, f"label_{split}.csv")))

    print(split, img_count, csv_count)

train 2116 2116
val 224 224
test 274 274


##Zip the folder

In [17]:
import shutil

source_path = "/content/brain_ct_project"
zip_path = "/content/air_filtered_data"

shutil.make_archive(zip_path, 'zip', source_path)

print("✅ Zip created at:", zip_path + ".zip")

✅ Zip created at: /content/air_filtered_data.zip


##Save to Google Drive

In [18]:
import shutil

drive_path = "/content/drive/MyDrive/brain_ct_project/air_filtered_data.zip"

shutil.move("/content/air_filtered_data.zip", drive_path)

print("✅ Zip moved to Drive:", drive_path)

✅ Zip moved to Drive: /content/drive/MyDrive/brain_ct_project/air_filtered_data.zip


In [19]:
import os

print(os.path.exists(drive_path))

True
